<a href="https://colab.research.google.com/github/abid2526/cse498R/blob/main/SL%26SD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Data

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("data.csv")

# Show first rows
print (df.head())

# Check Dataset

In [ ]:
print(df.info())
print(df["Sleep Disorder"].unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Person ID         5000 non-null   int64  
 1   Heart Rate        5000 non-null   int64  
 2   ECG_HeartRate     5000 non-null   int64  
 3   ECG_QRS_Duration  5000 non-null   float64
 4   EEG_Alpha         5000 non-null   float64
 5   EEG_Beta          5000 non-null   float64
 6   EEG_Theta         5000 non-null   float64
 7   EMG_Mean          5000 non-null   float64
 8   EMG_Max           5000 non-null   float64
 9   Stress Level      5000 non-null   int64  
 10  Sleep Disorder    5000 non-null   object 
dtypes: float64(6), int64(4), object(1)
memory usage: 429.8+ KB
None
['Insomnia' 'Sleep Apnea' 'REM Behavior Disorder' 'Narcolepsy']


# Data Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop unnecessary column
df = df.drop("Person ID", axis=1)

# Find text columns
categorical_columns = df.select_dtypes(include=['object']).columns

print("Categorical Columns:")
print(categorical_columns)

# Encode all text columns
le_dict = {}

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

print("\nData after encoding:")
print(df.head())

Categorical Columns:
Index(['Sleep Disorder'], dtype='object')

Data after encoding:
   Heart Rate  ECG_HeartRate  ECG_QRS_Duration  EEG_Alpha  EEG_Beta  \
0          77             98             0.112      82.49     37.26   
1          75             63             0.087      26.55     16.88   
2          75             76             0.100      31.89     49.30   
3          85             77             0.100      32.38      9.61   
4          85             93             0.088      92.12     24.45   

   EEG_Theta  EMG_Mean  EMG_Max  Stress Level  Sleep Disorder  
0      18.94     24.46    50.13             6               0  
1      22.35     67.15   152.81             8               3  
2      28.87     82.98   113.20             8               3  
3       5.92     62.65   124.20             8               3  
4      33.88     73.09   152.98             8               3  


# Slpit Features & Labels

In [ ]:
X = df.drop(["Stress Level", "Sleep Disorder"], axis=1)

y_stress = df["Stress Level"]
y_sleep = df["Sleep Disorder"]

In [ ]:
X = df.drop(["Stress Level", "Sleep Disorder"], axis=1)

y_stress = df["Stress Level"]
y_sleep = df["Sleep Disorder"]

# Train Stress Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_train, X_test, y_train, y_test = train_test_split(X, y_stress, test_size=0.2, random_state=42)

model_stress = RandomForestRegressor(
n_estimators=200, max_depth=10, random_state=42
)
model_stress.fit(X_train, y_train)

pred = model_stress.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))

MAE: 1.4558545359700814


# Train Sleep Disorder Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y_sleep, test_size=0.2, random_state=42)

model_sleep = RandomForestClassifier(
n_estimators=200, max_depth=10, random_state=42
)
model_sleep.fit(X_train, y_train)

pred = model_sleep.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# Save Models

In [ ]:
import joblib

joblib.dump(model_stress, "stress_model.pkl")
joblib.dump(model_sleep, "sleep_model.pkl")

['sleep_model.pkl']

# Test with new input

In [ ]:
import pandas as pd

# Sample format
sample = pd.DataFrame([[80, 95, 0.1, 30, 20, 15, 50, 120]],
                      columns=X.columns)

# Predict
stress_value = model_stress.predict(sample)[0]
sleep_value = model_sleep.predict(sample)[0]

# Convert sleep back to label
sleep_label = le.inverse_transform([sleep_value])[0]

# Print
print("Predicted Stress Level:", round(stress_value, 2))

if stress_value <= 3:
    print("Stress Category: Low")
elif stress_value <= 6:
    print("Stress Category: Moderate")
else:
    print("Stress Category: High")

print("Sleep Disorder:", sleep_label)

Predicted Stress Level: 5.43
Stress Category: Moderate
Sleep Disorder: narcolepsy


# Accuracy Check

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Predict
pred_stress = model_stress.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, pred_stress)
mse = mean_squared_error(y_test, pred_stress)
rmse = np.sqrt(mse)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)

MAE: 4.01828
MSE: 17.663897799999997
RMSE: 4.202844013284338
